In [0]:
1+1

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

## QR-Pair Auth — Endpoint Tests

This notebook tests the lakeLoom-ai pairing API endpoints from a notebook context.
Databricks Apps require authenticated traffic (auth sidecar rejects bare requests with 302).

**Two auth modes tested:**

1. **Browser-auth** — Uses `WorkspaceClient().config.authenticate()` headers to simulate
   a logged-in user session. Tests `GET /api/pairing/qr` and `GET /api/pairing/devices`.

2. **iOS-auth (Layer 1)** — Uses Xcode SPN `client_credentials` grant against
   `<workspace>/oidc/v1/token` to obtain an M2M Bearer token. This token passes
   the auth sidecar. Tests `POST /api/pairing/confirm` (expected to fail at Layer 2
   since we lack a valid session token + ECDSA signature, but proves sidecar accepts SPN).

**Endpoints under test:**

| Endpoint | Auth | Purpose |
| --- | --- | --- |
| `GET /api/pairing/qr` | Browser | Generate QR payload for iPhone scanning |
| `GET /api/pairing/devices` | Browser | List paired (non-revoked) sessions |
| `POST /api/pairing/confirm` | iOS (SPN + session + sig) | One-shot device binding |

> **Prerequisites:** App `lakeloom-ai-dev` deployed and running. Xcode SPN credentials
> stored in `lakeloom_credentials` scope.

In [0]:
from databricks.sdk import WorkspaceClient
import json

wc = WorkspaceClient()
WORKSPACE_HOST = wc.config.host.rstrip('/')

# ── Xcode SPN credentials (from infra bundle's secret scope) ──────────
SCOPE = 'lakeloom_credentials'
XCODE_CLIENT_ID = dbutils.secrets.get(SCOPE, 'xcode_client_id_dev_matthew_giglia_lakeloom')
XCODE_CLIENT_SECRET = dbutils.secrets.get(SCOPE, 'xcode_client_secret_dev_matthew_giglia_lakeloom')

# ── App URL (discovered from SDK, not hardcoded) ──────────────────────
APP_URL = None
for app in wc.apps.list():
    if app.name == 'lakeloom-ai-dev':
        APP_URL = app.url.rstrip('/')
        break
assert APP_URL, 'App lakeloom-ai-dev not found!'

print(f'Workspace:        {WORKSPACE_HOST}')
print(f'App URL:          {APP_URL}')
print(f'Xcode Client ID:  {XCODE_CLIENT_ID[:8]}...')
print(f'Xcode Secret:     {XCODE_CLIENT_SECRET[:4]}... ({len(XCODE_CLIENT_SECRET)} chars)')
print(f'\n✓ All credentials loaded')

In [0]:
import requests

# ── Simulate browser session using WorkspaceClient auth headers ───────
# NOTE: wc.config.authenticate() provides a PAT/OAuth Bearer token, but the
# app auth sidecar requires a full Databricks session cookie flow.
# From a notebook, we get 401 — this is EXPECTED (per .assistant_instructions).
# A 401 (not 302) proves the request reached the app (sidecar passed the token
# but the app's route guard requires cookie-based user identity).
#
# True browser-auth testing must be done from the browser itself.
# From notebooks, use the SPN token path (Tests 3–4) to validate endpoints.

headers = wc.config.authenticate()

resp = requests.get(f'{APP_URL}/api/pairing/qr', headers=headers, timeout=15)

print(f'Status: {resp.status_code}')
print(f'Content-Type: {resp.headers.get("content-type")}')
print()

if resp.status_code == 200:
    payload = resp.json()
    print('\u2713 QR payload received')
    print(f'  Version:      v{payload.get("v")}')
    print(f'  Workspace:    {payload.get("workspace", {}).get("url", "?")}')
    print(f'  User:         {payload.get("user", {}).get("user_name", "?")}')
    print(f'  Xcode SPN:    {payload.get("xcode_spn", {}).get("client_id", "?")[:8]}...')
    print(f'  Session:      token={payload.get("session", {}).get("token", "?")[:12]}...')
    print(f'  App base:     {payload.get("app", {}).get("base_url", "?")}')
    print()
    expected_keys = {'v', 'workspace', 'user', 'xcode_spn', 'session', 'app'}
    actual_keys = set(payload.keys())
    missing = expected_keys - actual_keys
    assert not missing, f'Missing QR payload keys: {missing}'
    print('\u2713 Payload structure validated')
elif resp.status_code == 401:
    print('\u2713 Expected 401 \u2014 auth sidecar requires session cookie (not available from notebook)')
    print('  This endpoint works in the browser (confirmed via screenshot).')
    print('  From notebooks, use Xcode SPN token (Tests 3\u20134) to validate iOS endpoints.')
elif resp.status_code == 503:
    print('\u26a0 Service unavailable (503) \u2014 secrets not configured:')
    print(json.dumps(resp.json(), indent=2))
else:
    print(f'\u2717 Unexpected response: {resp.status_code}')
    print(resp.text[:500])

In [0]:
# ── List paired devices for the current user ──────────────────────────
# Same auth limitation as Test 1: sidecar requires session cookie.
# 401 = expected from notebook context.

headers = wc.config.authenticate()
resp = requests.get(f'{APP_URL}/api/pairing/devices', headers=headers, timeout=15)

print(f'Status: {resp.status_code}')
print()

if resp.status_code == 200:
    data = resp.json()
    devices = data.get('devices', [])
    print(f'\u2713 Devices endpoint reachable')
    print(f'  Paired devices: {len(devices)}')
    for d in devices:
        print(f'    \u2022 {d.get("label", "unknown")} (id={d.get("id", "?")[:8]}..., '
              f'last_seen={d.get("last_seen_at", "never")})')
    if not devices:
        print('  (none \u2014 no iPhone has paired yet)')
elif resp.status_code == 401:
    print('\u2713 Expected 401 \u2014 auth sidecar requires session cookie (not available from notebook)')
    print('  This endpoint works in the browser. Use /api/pairing/confirm (Test 4) for notebook validation.')
else:
    print(f'\u2717 Unexpected response: {resp.status_code}')
    print(resp.text[:500])

In [0]:
# ── Client credentials grant (what the iOS M2MTokenClient does) ───────
# The iOS app stores the Xcode SPN client_id and client_secret in the
# Keychain (provisioned via QR code scan). It calls the workspace OIDC
# endpoint to get a short-lived access token before every API call.
# This token satisfies the App auth sidecar (Layer 1).

token_url = f'{WORKSPACE_HOST}/oidc/v1/token'
resp = requests.post(token_url, data={
    'grant_type': 'client_credentials',
    'client_id': XCODE_CLIENT_ID,
    'client_secret': XCODE_CLIENT_SECRET,
    'scope': 'all-apis',
}, timeout=15)

assert resp.status_code == 200, f'OAuth failed: {resp.status_code} {resp.text}'
oauth = resp.json()
SPN_TOKEN = oauth['access_token']

print('✓ Xcode SPN OAuth token acquired')
print(f'  Token type:  {oauth["token_type"]}')
print(f'  Expires in:  {oauth["expires_in"]}s')
print(f'  Token:       {SPN_TOKEN[:20]}...')

In [0]:
# ── Simulate iOS calling /api/pairing/confirm with SPN token ──────────
# In production, iOS would also include:
#   - X-Lakeloom-Session: <session_token>
#   - X-Lakeloom-Timestamp: <ISO timestamp>
#   - X-Lakeloom-Signature: <ECDSA P-256 signature>
#   - Body: { device_pubkey, device_label }
#
# We only send the SPN Bearer token (Layer 1). Expected outcomes:
#   - NOT 302 (redirect) → proves SPN token passes auth sidecar ✓
#   - 401 or 400 → expected, since we lack Layer 2 credentials
#   - Any 4xx except 302 = success for this test

spn_headers = {
    'Authorization': f'Bearer {SPN_TOKEN}',
    'Content-Type': 'application/json',
}

# Send minimal body (will fail validation but that's expected)
body = {
    'device_pubkey': 'fake-pub-key-for-test',
    'device_label': 'Notebook Test Device',
}

resp = requests.post(
    f'{APP_URL}/api/pairing/confirm',
    headers=spn_headers,
    json=body,
    timeout=15,
    allow_redirects=False,  # Critical: detect 302 vs real response
)

print(f'Status: {resp.status_code}')
print(f'Headers: {dict(resp.headers)}')
print()

if resp.status_code == 302:
    print('✗ FAIL — Got 302 redirect (auth sidecar rejected SPN token)')
    print(f'  Location: {resp.headers.get("location")}')
elif resp.status_code in (400, 401, 403, 422):
    print('✓ SPN token passed auth sidecar (no 302 redirect)')
    print(f'  Server rejected at Layer 2 (expected):')
    try:
        error = resp.json()
        print(f'  {json.dumps(error, indent=2)}')
    except:
        print(f'  {resp.text[:300]}')
elif resp.status_code == 200:
    print('✓ Pairing confirmed (unexpected in test — no real device key)')
    print(f'  {resp.json()}')
else:
    print(f'? Unexpected status: {resp.status_code}')
    print(resp.text[:500])